## ▶ What you'll see when you run this
- A **BEFORE vs AFTER** generation: `distilgpt2` produces rambling text, then the **LoRA-fine-tuned** model answers in the short `### Response:` style it was trained on.

**Time:** ~6 min · **Cost:** free (runs on CPU; a Colab T4 GPU is faster but optional) · **Keys:** none

# Week 16 · Notebook 1 — LoRA / PEFT Fine-Tuning Demo
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Fine-tune a **small open model** with **LoRA** (low-rank adapters) on a few instruction examples, using the Hugging Face stack — **no OpenAI**.

> **GPU optional:** this runs on CPU; for speed use Runtime → Change runtime type → **T4 GPU**.
> Every heavy import is guarded with try/except, so the notebook degrades gracefully (clear message) if a library is missing.

## 0. Install the Hugging Face fine-tuning stack
We install `peft` for LoRA. (We skip `bitsandbytes`/QLoRA here — it needs a CUDA GPU and won't load on CPU/Mac; see the QLoRA **concept-only** note in §3.)

In [ ]:
!pip -q install transformers datasets peft accelerate 2>/dev/null
print('installed (or already present)')

## 1. Check the environment
We report whether a GPU is available. This tiny demo trains fine on **CPU**; a GPU just makes it faster. (Real, large fine-tuning jobs do need a GPU.)

In [ ]:
HAS_GPU = False
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    if HAS_GPU:
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'GPU: {name}  ~{vram:.1f} GB VRAM')
    else:
        print('No GPU detected — fine. This small demo trains on CPU; a T4 GPU is just faster.')
except Exception as e:
    print('PyTorch not available yet:', e)

## 2. Memory math (why we use LoRA + quantization)
A rough rule for *loading* a model for inference:
- FP16 ~2 GB / billion params · 8-bit ~1 GB/B · 4-bit ~0.5 GB/B.
Training needs far more (gradients + optimizer + activations) — so we freeze the base model and train only small **LoRA adapters**.

In [ ]:
def loading_gb(params_billions, bits=16):
    return params_billions * (bits / 8) * 2  # ~2 bytes/param at 16-bit

for b in [16, 8, 4]:
    print(f'7B model @ {b}-bit: ~{loading_gb(7, b):.1f} GB to load')

## 3. A tiny instruction dataset (with a validation split)
Quality > quantity. Real projects use hundreds of clean, consistent examples; here we use a handful so the demo runs fast. We still hold out a couple of examples as a **validation split** so we can watch for overfitting.

> **QLoRA — concept only (GPU required).** QLoRA loads the *frozen base in 4-bit* (via `bitsandbytes` + `BitsAndBytesConfig`) and trains LoRA adapters on top, so even billion-parameter models fit one GPU. `bitsandbytes` needs CUDA and won't run on CPU/Mac, so this demo uses **plain LoRA**; QLoRA is the same code plus a `quantization_config=` argument on `from_pretrained`.

In [ ]:
all_examples = [
    {'instruction': 'Summarize in one sentence.',
     'input': 'The mitochondria is the powerhouse of the cell.',
     'output': 'Mitochondria produce most of the cell\'s energy.'},
    {'instruction': 'Summarize in one sentence.',
     'input': 'Python is a high-level, readable, general-purpose language.',
     'output': 'Python is a readable general-purpose programming language.'},
    {'instruction': 'Summarize in one sentence.',
     'input': 'A GPU runs many matrix operations in parallel.',
     'output': 'GPUs do parallel matrix math, ideal for neural networks.'},
    {'instruction': 'Summarize in one sentence.',
     'input': 'A vector database stores embeddings for similarity search.',
     'output': 'Vector databases index embeddings for fast similarity search.'},
    {'instruction': 'Summarize in one sentence.',
     'input': 'Transformers use attention to weigh relationships between tokens.',
     'output': 'Transformers use attention to relate tokens in a sequence.'},
]
# Hold out the last 2 examples as a validation split.
train_examples = all_examples[:3]
val_examples = all_examples[3:]
def format_example(ex):
    return (f"### Instruction: {ex['instruction']}\n"
            f"### Input: {ex['input']}\n"
            f"### Response: {ex['output']}")
print(f'{len(train_examples)} train · {len(val_examples)} val')
print(format_example(train_examples[0]))

## 4. Load a small base model + attach a LoRA adapter
We use **`distilgpt2`** — a small but **real** pretrained model (unlike a randomly initialized tiny model, it produces actual English, so fine-tuning visibly changes its style). The base weights are **frozen**; only the LoRA `A`/`B` matrices train (often <1% of params).

In [ ]:
MODEL_ID = 'distilgpt2'  # small but REAL pretrained model — meaningful output
model = tokenizer = None
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
    lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                      target_modules=['c_attn'], task_type='CAUSAL_LM')
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()  # see the <1% figure
except Exception as e:
    print('Could not load model/PEFT (libs missing?):', e)
    print('That is OK — the rest of the notebook still explains the workflow.')

## 4b. BEFORE: generate with the un-tuned model
Prompt the base model **before** fine-tuning. `distilgpt2` is a real model, so it writes fluent English — but it ignores our `### Response:` format and rambles. We'll compare this to the AFTER output in §5b.

In [ ]:
PROMPT = ('### Instruction: Summarize in one sentence.\n'
          '### Input: Neural networks learn patterns from data.\n'
          '### Response:')

def generate(m, prompt, max_new_tokens=40):
    if m is None or tokenizer is None:
        return '[model not loaded]'
    import torch
    ids = tokenizer(prompt, return_tensors='pt').to(m.device)
    with torch.no_grad():
        out = m.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][ids['input_ids'].shape[1]:],
                            skip_special_tokens=True).strip()

before = generate(model, PROMPT)
print('BEFORE fine-tuning:\n', before)

## 5. Tokenize and train for a couple of epochs
We keep `num_train_epochs` tiny so it finishes quickly. Watch the **loss** drop. We pass an **`eval_dataset`** (our validation split) so the Trainer reports **validation loss** each epoch — if it rises while training loss falls, you are **overfitting**.

Note: we **do not** pad to `max_length` here. The `DataCollatorForLanguageModeling` pads each batch dynamically and masks pad tokens in the labels, so the model isn't trained to predict padding.

In [ ]:
try:
    from datasets import Dataset
    from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
    def to_ds(examples):
        texts = [format_example(e) for e in examples]
        d = Dataset.from_dict({'text': texts})
        # No padding here — the collator pads each batch and masks pads in labels.
        return d.map(lambda b: tokenizer(b['text'], truncation=True, max_length=64),
                     batched=True, remove_columns=['text'])
    train_ds, val_ds = to_ds(train_examples), to_ds(val_examples)
    collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
    args = TrainingArguments(output_dir='out', num_train_epochs=8,
        per_device_train_batch_size=1, learning_rate=2e-4,
        logging_steps=1, eval_strategy='epoch', report_to=[])
    trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                      eval_dataset=val_ds, data_collator=collator)
    trainer.train()
    print('Training complete.')
except Exception as e:
    print('Skipping training (model/libs missing):', e)

## 5b. AFTER: generate with the fine-tuned model
Same prompt, now through the LoRA-adapted model. With only a few examples the change is modest, but you should see it lean toward the short, single-sentence `### Response:` style it was trained on — **that** is fine-tuning changing behavior.

In [ ]:
after = generate(model, PROMPT)
print('BEFORE:\n', before, '\n')
print('AFTER (LoRA fine-tuned):\n', after)

## 6. Save just the adapter (megabytes, not gigabytes)
You ship the small adapter and load it on top of the same base model. You can keep many task adapters for one base model.

In [ ]:
try:
    model.save_pretrained('my_lora_adapter')
    import os
    files = os.listdir('my_lora_adapter')
    print('Adapter saved:', files)
except Exception as e:
    print('Nothing to save (training did not run):', e)

## 7. Takeaways
- LoRA trains a tiny fraction of params and saves a small **adapter** — and it visibly nudged a real model's behavior (BEFORE vs AFTER).
- **QLoRA** (concept) loads the frozen base in 4-bit so even big models fit one GPU — same code plus a `quantization_config`, but it needs a CUDA GPU (not CPU/Mac).
- Fine-tune only after prompting and RAG fall short.
- Always keep a **validation split** and watch the **validation loss** for **overfitting**.